# IOM103 Coursework Source Code

## Sales Prediction and Customer Segmentation Analysis

This notebook implements the complete analytical workflow for Task A1 and Task B. It is designed to be executable from top to bottom and to reproduce the tables and figures discussed in the report.

| Block | Content | Dataset |
|---|---|---|
| Block 0 | Environment setup and helper functions | - |
| Block 1 | Task A1 data loading and preprocessing | Sales Dataset_Task A1.csv |
| Block 2 | Task A1 descriptive and correlation analysis | Sales Dataset_Task A1.csv |
| Block 3 | Task A1 supervised learning models | Sales Dataset_Task A1.csv |
| Block 4 | Task A1 feature importance | Sales Dataset_Task A1.csv |
| Block 5 | Task B data loading and exploration | Customer Segmentation_Task B 2.csv |
| Block 6 | Task B cluster validation and final K-Means | Customer Segmentation_Task B 2.csv |
| Block 7 | Final business summary | Both datasets |


---
# Block 0: Environment Setup and Helper Functions
This block imports the required libraries, defines the file-location helper, and configures the plotting style.


In [ ]:
# ============== Block 0: Environment Setup and Helper Functions ==============

# Basic libraries
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from cycler import cycler

# Display helper
from IPython.display import display

# Machine learning libraries - Task A1
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Machine learning libraries - Task B
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Plot settings
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["axes.prop_cycle"] = cycler(color=["#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974", "#64B5CD"])
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False

# File helper: this allows the notebook to run when datasets are in the same folder,
# or when the notebook is executed in the original /mnt/data environment.
def locate_file(filename):
    candidates = [
        Path.cwd() / filename,
        Path("/mnt/data") / filename
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Cannot find required file: {filename}")

SALES_FILE = locate_file("Sales Dataset_Task A1.csv")
CUSTOMER_FILE = locate_file("Customer Segmentation_Task B 2.csv")

print("✓ Environment setup completed")
print(f"✓ Sales dataset path: {SALES_FILE}")
print(f"✓ Customer dataset path: {CUSTOMER_FILE}")


---
# Block 1: Task A1 Data Loading and Preprocessing
This block loads the sales dataset, checks its structure, converts the date variable, and creates additional business variables for analysis.


In [ ]:
# ============== Block 1: Task A1 Data Loading and Preprocessing ==============

sales_df = pd.read_csv(SALES_FILE)

print("=" * 70)
print("Task A1 - Sales Dataset Loading")
print("=" * 70)
print(f"✓ Dataset shape: {sales_df.shape[0]} rows × {sales_df.shape[1]} columns")
print(f"✓ Original columns: {list(sales_df.columns)}")

print("\nMissing value check:")
print(sales_df.isnull().sum())

print("\nData preview:")
display(sales_df.head())


In [ ]:
# Date conversion and feature engineering
sales_df["Date"] = pd.to_datetime(
    sales_df["Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Time features for exploring seasonal or calendar-related effects
sales_df["Year"] = sales_df["Date"].dt.year
sales_df["Month"] = sales_df["Date"].dt.month
sales_df["Quarter"] = sales_df["Date"].dt.quarter
sales_df["Weekday"] = sales_df["Date"].dt.day_name()

# Business indicators for descriptive analysis
sales_df["Revenue"] = sales_df["Price"] * sales_df["Sales"]
sales_df["NetPrice"] = sales_df["Price"] * (1 - sales_df["Discount"] / 100)
sales_df["DiscountValue"] = sales_df["Price"] * sales_df["Discount"] / 100

print("=" * 70)
print("Task A1 - Preprocessing Summary")
print("=" * 70)
print(f"✓ Date range: {sales_df['Date'].min().date()} to {sales_df['Date'].max().date()}")
print(f"✓ Product categories: {sales_df['Product'].nunique()} -> {list(sales_df['Product'].unique())}")
print(f"✓ Customer types: {sales_df['CustomerType'].nunique()} -> {list(sales_df['CustomerType'].unique())}")
print(f"✓ Missing values after preprocessing: {sales_df.isnull().sum().sum()}")

display(sales_df.head())


---
# Block 2: Task A1 Descriptive and Correlation Analysis
This block calculates the main business metrics, compares products and customer types, and evaluates linear relationships between variables.


In [ ]:
# ============== Block 2: Task A1 Descriptive and Correlation Analysis ==============

print("=" * 70)
print("Core Business Metrics")
print("=" * 70)

core_metrics = {
    "Total Transactions": f"{len(sales_df):,}",
    "Total Units Sold": f"{sales_df['Sales'].sum():,}",
    "Total Gross Revenue": f"JPY {sales_df['Revenue'].sum():,.2f}",
    "Average Price": f"JPY {sales_df['Price'].mean():.2f}",
    "Average Discount Rate": f"{sales_df['Discount'].mean():.2f}%",
    "Average Marketing Effort": f"JPY {sales_df['MarketingEffort'].mean():,.2f}",
    "Average Sales per Transaction": f"{sales_df['Sales'].mean():.2f} units"
}

for key, value in core_metrics.items():
    print(f"{key:<32}: {value}")

print("\nNumeric variable summary:")
numeric_summary = sales_df[["Price", "Discount", "MarketingEffort", "Sales", "Revenue", "NetPrice"]].describe().round(2)
display(numeric_summary)


In [ ]:
# Product and customer-type summaries
product_summary = sales_df.groupby("Product").agg(
    Transactions=("Sales", "count"),
    TotalUnits=("Sales", "sum"),
    AvgSales=("Sales", "mean"),
    AvgPrice=("Price", "mean"),
    AvgDiscount=("Discount", "mean"),
    TotalRevenue=("Revenue", "sum")
).round(2).sort_values("TotalRevenue", ascending=False)

customer_type_summary = sales_df.groupby("CustomerType").agg(
    Transactions=("Sales", "count"),
    TotalUnits=("Sales", "sum"),
    AvgSales=("Sales", "mean"),
    AvgMarketing=("MarketingEffort", "mean"),
    TotalRevenue=("Revenue", "sum")
).round(2).sort_values("TotalRevenue", ascending=False)

print("=" * 70)
print("Product-Level Summary")
print("=" * 70)
display(product_summary)

print("=" * 70)
print("Customer-Type Summary")
print("=" * 70)
display(customer_type_summary)


In [ ]:
# Visual overview of sales performance
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.barplot(data=product_summary.reset_index(), x="TotalRevenue", y="Product", ax=axes[0, 0], color="#4C72B0")
axes[0, 0].set_title("Revenue by Product Category")
axes[0, 0].set_xlabel("Total Revenue")
axes[0, 0].set_ylabel("Product")

sns.barplot(data=customer_type_summary.reset_index(), x="CustomerType", y="TotalRevenue", ax=axes[0, 1], color="#4C72B0")
axes[0, 1].set_title("Revenue by Customer Type")
axes[0, 1].set_xlabel("Customer Type")
axes[0, 1].set_ylabel("Total Revenue")

sns.histplot(sales_df["Sales"], bins=25, kde=True, ax=axes[1, 0], color="#4C72B0")
axes[1, 0].set_title("Distribution of Sales Units")
axes[1, 0].set_xlabel("Sales")

sns.histplot(sales_df["Price"], bins=25, kde=True, ax=axes[1, 1], color="#4C72B0")
axes[1, 1].set_title("Distribution of Product Price")
axes[1, 1].set_xlabel("Price")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation analysis
corr_columns = ["Price", "Discount", "MarketingEffort", "Sales", "Revenue", "NetPrice", "DiscountValue"]
corr_matrix = sales_df[corr_columns].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Correlation Matrix of Sales Variables")
plt.tight_layout()
plt.show()

key_corr = pd.DataFrame({
    "Relationship": [
        "MarketingEffort vs Sales",
        "Price vs Sales",
        "Discount vs Sales",
        "Price vs Revenue",
        "NetPrice vs Revenue"
    ],
    "Correlation": [
        sales_df["MarketingEffort"].corr(sales_df["Sales"]),
        sales_df["Price"].corr(sales_df["Sales"]),
        sales_df["Discount"].corr(sales_df["Sales"]),
        sales_df["Price"].corr(sales_df["Revenue"]),
        sales_df["NetPrice"].corr(sales_df["Revenue"])
    ],
    "Interpretation": [
        "Negligible",
        "Negligible",
        "Very weak negative",
        "Very strong positive",
        "Strong positive"
    ]
})

print("=" * 70)
print("Key Correlation Coefficients")
print("=" * 70)
display(key_corr.round(3))


In [ ]:
# Marketing effectiveness exploration
sales_df["MarketingLevel"] = pd.qcut(
    sales_df["MarketingEffort"],
    q=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"]
)

marketing_summary = sales_df.groupby("MarketingLevel", observed=True).agg(
    Transactions=("Sales", "count"),
    AvgSales=("Sales", "mean"),
    AvgRevenue=("Revenue", "mean"),
    AvgMarketing=("MarketingEffort", "mean"),
    AvgDiscount=("Discount", "mean")
).round(2)

print("=" * 70)
print("Marketing Effort Binning Analysis")
print("=" * 70)
display(marketing_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=marketing_summary.reset_index(), x="MarketingLevel", y="AvgSales", ax=axes[0], color="#4C72B0")
axes[0].set_title("Average Sales by Marketing Level")
axes[0].set_xlabel("Marketing Effort Level")
axes[0].set_ylabel("Average Sales")
axes[0].tick_params(axis="x", rotation=25)

sns.scatterplot(data=sales_df, x="MarketingEffort", y="Sales", hue="CustomerType", alpha=0.65, ax=axes[1])
axes[1].set_title("Marketing Effort and Sales by Customer Type")
axes[1].set_xlabel("Marketing Effort")
axes[1].set_ylabel("Sales")

plt.tight_layout()
plt.show()


---
# Block 3: Task A1 Supervised Learning Models
This block develops Linear Regression, Regression Tree and Random Forest models to predict Sales. RMSE is used as the main performance metric.


In [ ]:
# ============== Block 3: Task A1 Supervised Learning Models ==============

# Select modelling variables. Product and CustomerType are categorical and will be one-hot encoded.
model_df = sales_df[
    ["Price", "Discount", "MarketingEffort", "Product", "CustomerType", "Month", "Quarter", "Sales"]
].copy()

model_df = pd.get_dummies(
    model_df,
    columns=["Product", "CustomerType"],
    drop_first=True
)

X = model_df.drop("Sales", axis=1)
y = model_df["Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# Standardisation supports fair numerical treatment and is necessary for linear regression comparability.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=" * 70)
print("Model Dataset Preparation")
print("=" * 70)
print(f"✓ Training set: {X_train.shape[0]} rows")
print(f"✓ Test set: {X_test.shape[0]} rows")
print(f"✓ Number of model features after encoding: {X_train.shape[1]}")
print("\nFeature list:")
print(list(X.columns))


In [ ]:
# Train and evaluate the supervised learning models
models = {
    "Mean Baseline": None,
    "Linear Regression": LinearRegression(),
    "Regression Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

model_results = []
trained_models = {}

for model_name, model in models.items():
    if model is None:
        # Baseline prediction: always predict the mean sales value from the training set
        train_pred = np.full(y_train.shape, y_train.mean())
        test_pred = np.full(y_test.shape, y_train.mean())
    else:
        model.fit(X_train_scaled, y_train)
        train_pred = model.predict(X_train_scaled)
        test_pred = model.predict(X_test_scaled)
        trained_models[model_name] = model

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    model_results.append({
        "Model": model_name,
        "Train RMSE": train_rmse,
        "Test RMSE": test_rmse,
        "RMSE Gap": test_rmse - train_rmse,
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Test R2": r2_score(y_test, test_pred)
    })

model_results_df = pd.DataFrame(model_results).round(4)

print("=" * 70)
print("Model Performance Comparison")
print("=" * 70)
display(model_results_df)

required_models = model_results_df[model_results_df["Model"].isin([
    "Linear Regression", "Regression Tree", "Random Forest"
])]
best_required_model = required_models.sort_values("Test RMSE").iloc[0]
print(f"Best required model based on Test RMSE: {best_required_model['Model']} ({best_required_model['Test RMSE']:.2f})")


In [ ]:
# Visual comparison of required models
required_models = model_results_df[model_results_df["Model"].isin([
    "Linear Regression", "Regression Tree", "Random Forest"
])]

rmse_plot_df = required_models.melt(
    id_vars="Model",
    value_vars=["Train RMSE", "Test RMSE"],
    var_name="Dataset",
    value_name="RMSE"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=rmse_plot_df, x="Model", y="RMSE", hue="Dataset", ax=axes[0])
axes[0].set_title("Train-Test RMSE Comparison")
axes[0].set_xlabel("Model")
axes[0].set_ylabel("RMSE")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=required_models, x="Model", y="RMSE Gap", ax=axes[1], color="#4C72B0")
axes[1].set_title("Generalisation Gap")
axes[1].set_xlabel("Model")
axes[1].set_ylabel("Test RMSE - Train RMSE")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


---
# Block 4: Task A1 Feature Importance
This block uses the Random Forest model to identify which variables contributed most to the prediction process.


In [ ]:
# ============== Block 4: Task A1 Feature Importance ==============

rf_model = trained_models["Random Forest"]
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("=" * 70)
print("Random Forest Feature Importance")
print("=" * 70)
display(feature_importance.round(4))

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x="Importance", y="Feature", color="#4C72B0")
plt.title("Top Random Forest Feature Importance Scores")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


---
# Block 5: Task B Data Loading and Exploration
This block loads the customer segmentation dataset and reviews the basic customer profile before clustering.


In [ ]:
# ============== Block 5: Task B Data Loading and Exploration ==============

customer_df = pd.read_csv(CUSTOMER_FILE)

print("=" * 70)
print("Task B - Customer Dataset Loading")
print("=" * 70)
print(f"✓ Dataset shape: {customer_df.shape[0]} rows × {customer_df.shape[1]} columns")
print(f"✓ Columns: {list(customer_df.columns)}")

print("\nMissing value check:")
print(customer_df.isnull().sum())

print("\nGender distribution:")
print(customer_df["Gender"].value_counts())

print("\nDescriptive statistics:")
display(customer_df.describe().round(2))

print("\nData preview:")
display(customer_df.head())


In [ ]:
# Exploratory charts for customer variables
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(customer_df["Age"], bins=20, kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")

sns.histplot(customer_df["Annual Income (k$)"], bins=20, kde=True, ax=axes[1], color="#4C72B0")
axes[1].set_title("Annual Income Distribution")
axes[1].set_xlabel("Annual Income (k$)")

sns.histplot(customer_df["Spending Score (1-100)"], bins=20, kde=True, ax=axes[2], color="#4C72B0")
axes[2].set_title("Spending Score Distribution")
axes[2].set_xlabel("Spending Score")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=customer_df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Gender",
    alpha=0.75
)
plt.title("Income and Spending Score by Gender")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score")
plt.tight_layout()
plt.show()


---
# Block 6: Task B Cluster Validation and Final K-Means Model
This block determines the number of clusters using the Elbow Method and Silhouette Score, then fits the final K-Means model.


In [ ]:
# ============== Block 6: Task B Cluster Validation and Final K-Means Model ==============

# Annual Income and Spending Score directly represent purchasing capacity and spending tendency.
cluster_features = customer_df[["Annual Income (k$)", "Spending Score (1-100)"]].copy()

inertia_values = []
silhouette_values = []

for k in range(2, 11):
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_temp = kmeans_temp.fit_predict(cluster_features)
    inertia_values.append(kmeans_temp.inertia_)
    silhouette_values.append(silhouette_score(cluster_features, labels_temp))

cluster_eval_df = pd.DataFrame({
    "K": range(2, 11),
    "Inertia": inertia_values,
    "Silhouette Score": silhouette_values
}).round(4)

print("=" * 70)
print("Cluster Number Evaluation")
print("=" * 70)
display(cluster_eval_df)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(data=cluster_eval_df, x="K", y="Inertia", marker="o", ax=axes[0], color="#4C72B0")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of Clusters")
axes[0].set_ylabel("Inertia")

sns.lineplot(data=cluster_eval_df, x="K", y="Silhouette Score", marker="o", ax=axes[1], color="#55A868")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of Clusters")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()


In [ ]:
# Final K-Means model with K=5
final_k = 5
kmeans_model = KMeans(n_clusters=final_k, random_state=42, n_init=10)
customer_df["Cluster"] = kmeans_model.fit_predict(cluster_features)

segment_names = {
    0: "Central Mainstream Customers",
    1: "Priority High-Spenders",
    2: "Young Promotion Responders",
    3: "Affluent Low-Engagement Customers",
    4: "Price-Sensitive Minimal Spenders"
}
customer_df["Segment Name"] = customer_df["Cluster"].map(segment_names)

cluster_centers = pd.DataFrame(
    kmeans_model.cluster_centers_,
    columns=["Annual Income (k$)", "Spending Score (1-100)"]
).round(2)

cluster_profile = customer_df.groupby(["Cluster", "Segment Name"]).agg(
    Count=("CustomerID", "count"),
    AvgAge=("Age", "mean"),
    AvgIncome=("Annual Income (k$)", "mean"),
    AvgSpendingScore=("Spending Score (1-100)", "mean"),
    Female=("Gender", lambda x: (x == "Female").sum()),
    Male=("Gender", lambda x: (x == "Male").sum())
).round(2)

print("=" * 70)
print("Final Cluster Centers")
print("=" * 70)
display(cluster_centers)

print("=" * 70)
print("Customer Segment Profiles")
print("=" * 70)
display(cluster_profile)


In [ ]:
# Customer segmentation visualisation
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.scatterplot(
    data=customer_df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Segment Name",
    s=75,
    alpha=0.8,
    ax=axes[0, 0]
)
axes[0, 0].scatter(
    cluster_centers["Annual Income (k$)"],
    cluster_centers["Spending Score (1-100)"],
    marker="X",
    s=250,
    c="black",
    label="Centroids"
)
axes[0, 0].set_title("Customer Segments in Income-Spending Space")
axes[0, 0].legend(fontsize=7, loc="best")

cluster_size_df = customer_df["Segment Name"].value_counts().reset_index()
cluster_size_df.columns = ["Segment Name", "Count"]
sns.barplot(data=cluster_size_df, x="Count", y="Segment Name", ax=axes[0, 1], color="#4C72B0")
axes[0, 1].set_title("Segment Size Distribution")

profile_plot_df = cluster_profile.reset_index().melt(
    id_vars=["Cluster", "Segment Name"],
    value_vars=["AvgAge", "AvgIncome", "AvgSpendingScore"],
    var_name="Metric",
    value_name="Average"
)
sns.barplot(data=profile_plot_df, x="Average", y="Segment Name", hue="Metric", ax=axes[1, 0])
axes[1, 0].set_title("Average Segment Characteristics")

segment_gender_df = customer_df.groupby(["Segment Name", "Gender"]).size().reset_index(name="Count")
sns.barplot(data=segment_gender_df, x="Count", y="Segment Name", hue="Gender", ax=axes[1, 1])
axes[1, 1].set_title("Gender Composition by Segment")

plt.tight_layout()
plt.show()


---
# Block 7: Final Business Summary
This block prints the final interpretation that links the technical outputs to business decisions.


In [ ]:
# ============== Block 7: Final Business Summary ==============

print("=" * 70)
print("Final Business Summary")
print("=" * 70)

summary_text = f"""
Task A1 - Sales Prediction
1. The sales dataset contains {len(sales_df):,} transactions from {sales_df['Date'].min().date()} to {sales_df['Date'].max().date()}.
2. Total gross revenue is JPY {sales_df['Revenue'].sum():,.2f}, with {sales_df['Sales'].sum():,} units sold.
3. MarketingEffort has a very weak direct correlation with Sales: {sales_df['MarketingEffort'].corr(sales_df['Sales']):.3f}.
4. Among Linear Regression, Regression Tree and Random Forest, Linear Regression has the lowest test RMSE.
5. The Regression Tree overfits heavily, while Random Forest still shows a sizeable train-test gap.

Task B - Customer Segmentation
1. Annual Income and Spending Score were used as the clustering variables.
2. K=5 is selected because it provides the best Silhouette Score and a clear elbow pattern.
3. The five segments are Central Mainstream Customers, Priority High-Spenders,
   Young Promotion Responders, Affluent Low-Engagement Customers, and Price-Sensitive Minimal Spenders.
4. Priority High-Spenders and Young Promotion Responders should be prioritised for promotion events.
5. Affluent Low-Engagement Customers should be activated with personalised premium recommendations.
"""

print(summary_text)
